In [0]:
import json
import sys
from pathlib import Path

dbutils.widgets.text("repo_root", "")
supplied = dbutils.widgets.get("repo_root").strip()

if supplied:
    REPO_ROOT = Path(supplied).resolve()
else:
    REPO_ROOT = next(
        (
            candidate
            for candidate in [Path.cwd(), *Path.cwd().parents]
            if (candidate / "config/runtime.yaml").is_file()
            and (candidate / "scripts/run_config.py").is_file()
        ),
        None,
    )

if REPO_ROOT is None:
    raise ValueError(
        "Set the repo_root widget to your Workspace repository root."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.run_config import notebook_context

REPO_ROOT, config, runtime, controls = notebook_context(
    spark,
    dbutils,
)

print("Environment:", config["environment_name"])
print("Namespace:", config["namespace"])

In [0]:
from scripts.layer_runner import (
    active_context,
    run_delivery_layer,
    finish_delivery,
)

context_json = dbutils.widgets.get("delivery_context").strip()

delivery = (
    json.loads(context_json)
    if context_json
    else active_context(spark, config, controls)
)

gold_result = run_delivery_layer(
    spark,
    config,
    controls,
    delivery,
    layer="GOLD",
)

completion_result = finish_delivery(
    spark,
    config,
    controls,
    delivery,
)

print(
    json.dumps(
        {
            "delivery": delivery,
            "gold": gold_result,
            "completion": completion_result,
        },
        default=str,
        indent=2,
    )
)